# Notebook 06 — Enriquecimento de Temas e Votações

**Sprint 4.2 — Lei e Política**

A coleta da Sprint 1 guardou apenas proposições dos tipos `PL/PEC/PLP/MPV/PDL`
apresentadas em 2025–2026. Por isso **95 das 137 votações nominais do plenário**
ficaram sem proposição linkada (`votacoes.proposicao_id = NULL`): elas referenciam
projetos mais antigos (PLs de 2004–2017 ainda em pauta) ou requerimentos que não
entraram no corpus. Sem proposição não há tema — e ~30 mil votos individuais ficavam
fora do perfil cidadão e do treino do modelo.

Este notebook é **incremental e idempotente**:
1. Identifica as votações órfãs e os ids das proposições que elas referenciam
2. Coleta essas proposições faltantes na API da Câmara (`/proposicoes/{id}`)
3. Classifica cada uma em um dos 10 temas **já existentes** — sem re-clusterizar:
   reajusta o mesmo TF-IDF + K-Means da Sprint 2 sobre o corpus já rotulado, deriva o
   mapa `cluster → tema_cidadao` dos próprios dados (robusto a renumeração) e **prevê**
   o cluster das novas proposições.
4. Religa `votacoes.proposicao_id` (backfill por prefixo do id da votação)

Depende de: Sprint 1 (coleta) e Sprint 2 (clustering) já executadas.

In [1]:
import sys
sys.path.insert(0, '..')

import re
import logging
import unicodedata

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

from src.coleta import _get_com_retry, BASE_CAMARA, salvar_raw
from src.db import buscar_todos, upsert_proposicoes, get_client
from src.classificacao.natureza import classificar_natureza, ROTULO_NATUREZA

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('06_enriquecimento')
print('Módulos carregados.')

Módulos carregados.


## 1. Votações órfãs e proposições faltantes

O `id_externo` da votação tem o formato `"<id_proposicao>-<n>"` (ex.: `"2484059-7"`).
O prefixo é o `id_externo` da proposição. Listamos os prefixos das votações órfãs e
descobrimos quais ainda não estão na tabela `proposicoes`.

In [2]:
vot_db = buscar_todos('votacoes', 'id,id_externo,proposicao_id')
orfas = [v for v in vot_db if v.get('proposicao_id') is None]

def prefixo_prop(id_ext):
    try:
        return int(str(id_ext).split('-')[0])
    except (ValueError, AttributeError):
        return None

prefixos = {p for p in (prefixo_prop(v['id_externo']) for v in orfas) if p is not None}

prop_db = buscar_todos('proposicoes', 'id_externo,casa')
ja_no_banco = {int(p['id_externo']) for p in prop_db if p['casa'] == 'camara'}

faltantes = sorted(prefixos - ja_no_banco)
print(f'Votações órfãs: {len(orfas)}')
print(f'Proposições distintas referenciadas: {len(prefixos)}')
print(f'Já no banco: {len(prefixos & ja_no_banco)} | A coletar: {len(faltantes)}')

19:18:03 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votacoes?select=id%2Cid_externo%2Cproposicao_id&offset=0&limit=1000 "HTTP/2 200 OK"


19:18:04 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa&offset=0&limit=1000 "HTTP/2 200 OK"


19:18:05 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa&offset=1000&limit=1000 "HTTP/2 200 OK"


19:18:05 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa&offset=2000&limit=1000 "HTTP/2 200 OK"


19:18:05 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa&offset=3000&limit=1000 "HTTP/2 200 OK"


...

19:18:16 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa&offset=19000&limit=1000 "HTTP/2 200 OK"


19:18:16 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa&offset=20000&limit=1000 "HTTP/2 200 OK"


19:18:17 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa&offset=21000&limit=1000 "HTTP/2 200 OK"


19:18:17 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa&offset=22000&limit=1000 "HTTP/2 200 OK"


Votações órfãs: 0
Proposições distintas referenciadas: 0
Já no banco: 0 | A coletar: 0


## 2. Coletar as proposições faltantes

Uma chamada `GET /proposicoes/{id}` por proposição (respeitando o rate-limiter global).
Gravamos com `upsert_proposicoes` — ainda **sem tema**; o tema é atribuído na etapa 3.

In [3]:
def coletar_proposicao(pid):
    resp = _get_com_retry(f'{BASE_CAMARA}/proposicoes/{pid}')
    if resp.status_code != 200:
        return None
    return resp.json().get('dados')

def data_segura(s):
    if not s:
        return None
    m = re.match(r'(\d{4}-\d{2}-\d{2})', str(s))
    return m.group(1) if m else None

brutos, registros_prop, falhas = [], [], []
for pid in faltantes:
    d = coletar_proposicao(pid)
    if not d:
        falhas.append(pid)
        continue
    brutos.append(d)
    ementa = (d.get('ementa') or '').strip()
    if len(ementa) < 10:
        ementa = (d.get('ementa') or d.get('keywords') or '(sem ementa)').strip()
    registros_prop.append({
        'id_externo': int(d['id']),
        'casa': 'camara',
        'ementa': ementa,
        'keywords': d.get('keywords', '') or '',
        'data': data_segura(d.get('dataApresentacao')),
    })

salvar_raw('camara_proposicoes_enriquecimento.json', brutos)
print(f'Coletadas: {len(registros_prop)} | Falhas: {len(falhas)} {falhas[:10]}')

total = upsert_proposicoes(registros_prop)
print(f'Proposições inseridas/atualizadas: {total}')

19:18:17 [INFO] Salvo: /home/brandao/Documentos/trabalhoCiênciadeDados/data/raw/camara_proposicoes_enriquecimento.json (0 registros)


Coletadas: 0 | Falhas: 0 []
Proposições inseridas/atualizadas: 0


## 3. Classificar nas 10 categorias existentes (sem re-clusterizar)

Reproduzimos a **mesma limpeza** da Sprint 2 (notebook 03), reajustamos TF-IDF + K-Means
sobre o corpus **já rotulado** e derivamos o mapa `cluster → tema_cidadao` por **moda**
do tema armazenado em cada cluster (robusto a renumeração do K-Means). Em seguida
`transform` + `predict` atribuem o tema das novas proposições. As categorias originais
das 22 mil proposições **não são alteradas**.

In [4]:
def remover_acentos(texto):
    nfkd = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

STOPWORDS_PT = {
    'a', 'o', 'as', 'os', 'um', 'uma', 'uns', 'umas', 'de', 'do', 'da', 'dos',
    'das', 'em', 'no', 'na', 'nos', 'nas', 'por', 'pelo', 'pela', 'pelos',
    'pelas', 'com', 'sem', 'sob', 'sobre', 'para', 'pra', 'ate', 'entre',
    'contra', 'desde', 'e', 'ou', 'mas', 'que', 'se', 'como', 'quando',
    'porque', 'pois', 'ja', 'nao', 'sim', 'ao', 'aos', 'este', 'esta', 'estes',
    'estas', 'esse', 'essa', 'esses', 'essas', 'isto', 'isso', 'aquele',
    'aquela', 'aquilo', 'seu', 'sua', 'seus', 'suas', 'dele', 'dela', 'deles',
    'delas', 'meu', 'minha', 'nosso', 'nossa', 'ele', 'ela', 'eles', 'elas',
    'eu', 'tu', 'voce', 'nos', 'vos', 'lhe', 'lhes', 'me', 'te', 'foi', 'ser',
    'sao', 'era', 'sera', 'tem', 'ter', 'havia', 'mais', 'menos', 'muito',
    'pouco', 'todo', 'toda', 'todos', 'todas', 'outro', 'outra', 'outros',
    'outras', 'mesmo', 'mesma', 'qual', 'quais', 'onde', 'seja', 'sejam',
    'tambem', 'apenas', 'cada', 'ainda', 'assim', 'entao',
}
STOPWORDS_JURIDICAS = {
    'lei', 'leis', 'art', 'arts', 'artigo', 'artigos', 'paragrafo', 'inciso',
    'alinea', 'dispoe', 'dispor', 'altera', 'alteracao', 'alterar', 'institui',
    'instituir', 'estabelece', 'estabelecer', 'providencias', 'outras', 'revoga',
    'revogacao', 'vigencia', 'dar', 'acrescenta', 'inclui', 'inclusao',
    'modifica', 'denomina', 'denominacao', 'autoriza', 'autorizacao', 'cria',
    'criacao', 'federal', 'nacional', 'numero', 'decreto', 'medida', 'provisoria',
    'projeto', 'proposta', 'emenda', 'constituicao', 'codigo', 'normas', 'norma',
    'regula', 'regulamenta', 'define', 'fixa', 'concede', 'referente', 'relativo',
    'relativa', 'seguinte', 'seguintes', 'forma', 'âmbito', 'ambito',
}
MESES = {'janeiro','fevereiro','marco','abril','maio','junho','julho',
         'agosto','setembro','outubro','novembro','dezembro'}
STOPWORDS = STOPWORDS_PT | STOPWORDS_JURIDICAS | MESES
try:
    from nltk.corpus import stopwords as _nltk_sw
    STOPWORDS |= {remover_acentos(w) for w in _nltk_sw.words('portuguese')}
except Exception:
    pass

def limpar(texto):
    texto = remover_acentos(str(texto).lower())
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    return ' '.join(t for t in texto.split() if len(t) >= 3 and t not in STOPWORDS)

In [5]:
# Reajusta o modelo do EIXO B (Sprint 2) sobre o corpus A3 já rotulado.
# Mesma config do nb03 (sublinear_tf, max_df, min_df) para reproduzir os clusters
# temáticos e derivar o mapa cluster->tema por moda (robusto a renumeração).
rot = pd.DataFrame(buscar_todos('proposicoes',
        'id_externo,casa,ementa,eixo,tema_cluster,tema_cidadao'))
rot = rot[(rot['eixo'] == 'tema') & (rot['tema_cluster'].notna())].copy()
rot['texto_limpo'] = rot['ementa'].map(limpar)
rot = rot[rot['texto_limpo'].str.len() > 0].copy()
print(f'Corpus A3 rotulado p/ reajuste: {len(rot)}')

vectorizer = TfidfVectorizer(sublinear_tf=True, max_df=0.30, min_df=8,
                             ngram_range=(1, 2), max_features=6000)
X_rot = vectorizer.fit_transform(rot['texto_limpo'])

K = int(rot['tema_cluster'].nunique())
km = KMeans(n_clusters=K, random_state=42, n_init=10)
rot['cluster_refit'] = km.fit_predict(X_rot)

# Mapas robustos derivados dos dados (moda do tema em cada cluster reajustado)
mapa_tema = (rot.groupby('cluster_refit')['tema_cidadao']
               .agg(lambda s: s.value_counts().idxmax()).to_dict())
mapa_idx = (rot.groupby('cluster_refit')['tema_cluster']
              .agg(lambda s: int(s.value_counts().idxmax())).to_dict())

pureza = (rot.groupby('cluster_refit')['tema_cidadao']
            .agg(lambda s: s.value_counts(normalize=True).max()).mean())
print(f'Pureza média da reprodução: {pureza:.3f}  (1.0 = reproduz a Sprint 2)')

19:18:18 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Ceixo%2Ctema_cluster%2Ctema_cidadao&offset=0&limit=1000 "HTTP/2 200 OK"


19:18:19 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Ceixo%2Ctema_cluster%2Ctema_cidadao&offset=1000&limit=1000 "HTTP/2 200 OK"


19:18:19 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Ceixo%2Ctema_cluster%2Ctema_cidadao&offset=2000&limit=1000 "HTTP/2 200 OK"


19:18:20 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Ceixo%2Ctema_cluster%2Ctema_cidadao&offset=3000&limit=1000 "HTTP/2 200 OK"


19:18:21 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Ceixo%2Ctema_cluster%2Ctema_cidadao&offset=4000&limit=1000 "HTTP/2 200 OK"


...

19:18:33 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Ceixo%2Ctema_cluster%2Ctema_cidadao&offset=20000&limit=1000 "HTTP/2 200 OK"


19:18:33 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Ceixo%2Ctema_cluster%2Ctema_cidadao&offset=21000&limit=1000 "HTTP/2 200 OK"


19:18:34 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Ceixo%2Ctema_cluster%2Ctema_cidadao&offset=22000&limit=1000 "HTTP/2 200 OK"


Corpus A3 rotulado p/ reajuste: 14091


Pureza média da reprodução: 0.825  (1.0 = reproduz a Sprint 2)


In [6]:
# Classifica proposições recém-coletadas pelos DOIS EIXOS.
# "Nova" = ainda sem natureza_codigo (nunca codificada). NÃO usar tema_cluster
# como sinal: A0/A1/A2 têm tema_cluster NULL por design (eixo de forma).
novas = pd.DataFrame(buscar_todos('proposicoes',
        'id_externo,casa,ementa,natureza_codigo'))
novas = novas[novas['natureza_codigo'].isna()].copy()

if novas.empty:
    print('Nenhuma proposição nova para classificar (todas já codificadas nos dois eixos).')
else:
    # Eixo A — natureza por regras
    nat = novas['ementa'].map(classificar_natureza)
    novas['natureza_codigo'] = nat.map(lambda x: x[0])
    novas['eixo'] = np.where(novas['natureza_codigo'] == 'A3_Substantiva', 'tema', 'forma')
    novas['tema_cidadao'] = novas['natureza_codigo'].map(ROTULO_NATUREZA)
    novas['tema_cluster'] = np.nan

    # Eixo B — só as A3 substantivas vão ao K-Means
    a3n = novas[novas['natureza_codigo'] == 'A3_Substantiva'].copy()
    a3n['texto_limpo'] = a3n['ementa'].map(limpar)
    a3n = a3n[a3n['texto_limpo'].str.len() > 0]
    if not a3n.empty:
        cl = pd.Series(km.predict(vectorizer.transform(a3n['texto_limpo'])), index=a3n.index)
        novas.loc[a3n.index, 'tema_cidadao'] = cl.map(mapa_tema)
        novas.loc[a3n.index, 'tema_cluster'] = cl.map(mapa_idx)

    print(f'Novas classificadas: {len(novas)}')
    print(novas['natureza_codigo'].value_counts())

    registros_tema = [
        {'id_externo': int(r['id_externo']), 'casa': r['casa'],
         'natureza_codigo': r['natureza_codigo'], 'eixo': r['eixo'],
         'tema_cluster': (int(r['tema_cluster']) if pd.notna(r['tema_cluster']) else None),
         'tema_cidadao': r['tema_cidadao']}
        for _, r in novas.iterrows()
    ]
    upsert_proposicoes(registros_tema)
    print('\nTemas gravados nas novas proposições (dois eixos).')

19:18:40 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Cnatureza_codigo&offset=0&limit=1000 "HTTP/2 200 OK"


19:18:41 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Cnatureza_codigo&offset=1000&limit=1000 "HTTP/2 200 OK"


19:18:42 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Cnatureza_codigo&offset=2000&limit=1000 "HTTP/2 200 OK"


19:18:42 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Cnatureza_codigo&offset=3000&limit=1000 "HTTP/2 200 OK"


19:18:43 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Cnatureza_codigo&offset=4000&limit=1000 "HTTP/2 200 OK"


...

19:18:53 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Cnatureza_codigo&offset=19000&limit=1000 "HTTP/2 200 OK"


19:18:53 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Cnatureza_codigo&offset=20000&limit=1000 "HTTP/2 200 OK"


19:18:54 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Cnatureza_codigo&offset=21000&limit=1000 "HTTP/2 200 OK"


19:18:55 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id_externo%2Ccasa%2Cementa%2Cnatureza_codigo&offset=22000&limit=1000 "HTTP/2 200 OK"


Nenhuma proposição nova para classificar (todas já codificadas nos dois eixos).


## 4. Backfill — religar `votacoes.proposicao_id`

Mesma lógica idempotente do notebook 01: para cada votação ainda sem proposição,
resolve o prefixo do `id_externo` contra o mapa atualizado de proposições.

In [7]:
prop_fresh = buscar_todos('proposicoes', 'id,id_externo,casa')
mapa_prop = {(int(r['id_externo']), r['casa']): r['id'] for r in prop_fresh}

vot_db = buscar_todos('votacoes', 'id,id_externo,proposicao_id')
sem_prop = [v for v in vot_db if v.get('proposicao_id') is None]
print(f'Votações sem proposicao_id antes: {len(sem_prop)}/{len(vot_db)}')

client = get_client()
atualizadas = nao_encontradas = 0
for v in sem_prop:
    try:
        pref = int(str(v['id_externo']).split('-')[0])
    except (ValueError, AttributeError):
        nao_encontradas += 1
        continue
    prop_id = mapa_prop.get((pref, 'camara'))
    if prop_id is None:
        nao_encontradas += 1
        continue
    client.table('votacoes').update({'proposicao_id': prop_id}).eq('id', v['id']).execute()
    atualizadas += 1

print(f'proposicao_id preenchido em: {atualizadas} votações')
print(f'Ainda sem proposição:        {nao_encontradas}')

19:18:55 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa&offset=0&limit=1000 "HTTP/2 200 OK"


19:18:55 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa&offset=1000&limit=1000 "HTTP/2 200 OK"


19:18:55 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa&offset=2000&limit=1000 "HTTP/2 200 OK"


19:18:56 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa&offset=3000&limit=1000 "HTTP/2 200 OK"


19:18:56 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa&offset=4000&limit=1000 "HTTP/2 200 OK"


...

19:19:01 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa&offset=20000&limit=1000 "HTTP/2 200 OK"


19:19:01 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa&offset=21000&limit=1000 "HTTP/2 200 OK"


19:19:02 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa&offset=22000&limit=1000 "HTTP/2 200 OK"


19:19:02 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votacoes?select=id%2Cid_externo%2Cproposicao_id&offset=0&limit=1000 "HTTP/2 200 OK"


Votações sem proposicao_id antes: 0/137
proposicao_id preenchido em: 0 votações
Ainda sem proposição:        0


## 5. Sanidade — votações com tema por categoria

In [8]:
vot = pd.DataFrame(buscar_todos('votacoes', 'id,proposicao_id'))
prop = pd.DataFrame(buscar_todos('proposicoes', 'id,tema_cidadao'))
m = vot.merge(prop, left_on='proposicao_id', right_on='id', how='left', suffixes=('', '_p'))

com_tema = m['tema_cidadao'].notna().sum()
print(f'Votações com tema: {com_tema}/{len(vot)}\n')
print('Votações por tema (categorias com votação nominal):')
print(m[m['tema_cidadao'].notna()]['tema_cidadao'].value_counts())

19:19:02 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votacoes?select=id%2Cproposicao_id&offset=0&limit=1000 "HTTP/2 200 OK"


19:19:02 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=0&limit=1000 "HTTP/2 200 OK"


19:19:03 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=1000&limit=1000 "HTTP/2 200 OK"


19:19:03 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=2000&limit=1000 "HTTP/2 200 OK"


19:19:03 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=3000&limit=1000 "HTTP/2 200 OK"


...

19:19:08 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=19000&limit=1000 "HTTP/2 200 OK"


19:19:08 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=20000&limit=1000 "HTTP/2 200 OK"


19:19:09 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=21000&limit=1000 "HTTP/2 200 OK"


19:19:09 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=22000&limit=1000 "HTTP/2 200 OK"


Votações com tema: 137/137

Votações por tema (categorias com votação nominal):
tema_cidadao
Outras Políticas Públicas                   65
Requerimentos e Atos Internos               23
Tributação e Reforma Tributária              9
Datas Comemorativas                          6
Créditos Orçamentários                       5
Previdência e Assistência Social             5
Direito Penal e Crimes                       5
Administração e Serviços Públicos            4
Violência Doméstica e Direitos da Mulher     4
Operações de Crédito de Municípios           3
Políticas de Prevenção e Incentivo           3
Criança e Adolescente                        3
Segurança Pública                            2
Name: count, dtype: int64
